In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/FundamentofDS/processed/multi/ptbxl_labeled (multi).csv')

In [3]:
df.head(5)

,ecg_id,patient_id,age,sex,height,weight,strat_fold,filename_lr,filename_hr,scp_codes,superclasses,NORM,MI,STTC,CD,HYP
0,1,15709.0,56.0,1,NaN,63.0,3,records100/00000/00001_lr,records500/00000/00001_hr,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",['NORM'],1,0,0,0,0
1,2,13243.0,19.0,0,NaN,70.0,2,records100/00000/00002_lr,records500/00000/00002_hr,"{'NORM': 80.0, 'SBRAD': 0.0}",['NORM'],1,0,0,0,0
2,3,20372.0,37.0,1,NaN,69.0,5,records100/00000/00003_lr,records500/00000/00003_hr,"{'NORM': 100.0, 'SR': 0.0}",['NORM'],1,0,0,0,0
3,4,17014.0,24.0,0,NaN,82.0,3,records100/00000/00004_lr,records500/00000/00004_hr,"{'NORM': 100.0, 'SR': 0.0}",['NORM'],1,0,0,0,0
4,5,17448.0,19.0,1,NaN,70.0,4,records100/00000/00005_lr,records500/00000/00005_hr,"{'NORM': 100.0, 'SR': 0.0}",['NORM'],1,0,0,0,0


In [4]:
!pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.5 MB/s eta 0:00:00


In [5]:
import os
import numpy as np
import wfdb
from scipy import signal
record_path = df.iloc[0]['filename_lr']
rec_dir = os.path.dirname(record_path)
rec_name = os.path.basename(record_path)
raw_signal, _ = wfdb.rdsamp(rec_name, pn_dir=f"ptb-xl/1.0.3/{rec_dir}")

fs = 100
lead_idx = 1
time_axis = np.linspace(0, len(raw_signal) / fs, len(raw_signal))

def remove_spikes(sig, threshold_factor=4.5):
    cleaned_sig = sig.copy()
    med = np.median(sig, axis=0)
    mad = np.median(np.abs(sig - med), axis=0)

    threshold = threshold_factor * (mad + 1e-6)
    is_spike = np.abs(sig - med) > threshold

    med_filtered = signal.medfilt(sig, kernel_size=(5, 1))
    cleaned_sig[is_spike] = med_filtered[is_spike]
    return cleaned_sig

despiked_signal = remove_spikes(raw_signal, threshold_factor=4.5)

In [6]:
lowcut, highcut = 0.5, 40.0
nyq = 0.5 * fs
b, a = signal.butter(2, [lowcut/nyq, highcut/nyq], btype='band')
filtered_signal = signal.filtfilt(b, a, despiked_signal, axis=0)

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
final_signal = scaler.fit_transform(filtered_signal)

In [8]:
from tqdm import tqdm
def preprocess_ecg_signal(raw_signal, fs=100):
    despiked = remove_spikes(raw_signal, threshold_factor=4.5)

    lowcut, highcut = 0.5, 40.0
    nyq = 0.5 * fs
    b, a = signal.butter(2, [lowcut/nyq, highcut/nyq], btype='band')
    filtered = signal.filtfilt(b, a, despiked, axis=0)

    scaler = StandardScaler()
    normalized = scaler.fit_transform(filtered)

    return normalized

In [9]:
print(df.columns.tolist())

['ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 'strat_fold', 'filename_lr', 'filename_hr', 'scp_codes', 'superclasses', 'NORM', 'MI', 'STTC', 'CD', 'HYP']


In [10]:
label_cols = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
y = df[label_cols].values

X = []

print(f"Total records to process: {len(df)}")
print(f"Label shape: {y.shape}")

Total records to process: 21799
Label shape: (21799, 5)


In [11]:
data_dir = '/content/drive/MyDrive/FundamentofDS'
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing ECG records"):
    full_path = os.path.join(data_dir, row['filename_lr'])
    raw_signal, _ = wfdb.rdsamp(full_path)

    processed_signal = preprocess_ecg_signal(raw_signal, fs=100)
    X.append(processed_signal)

X = np.array(X)

print("\nCompleted!")
print(f"X feature matrix shape: {X.shape}")
print(f"y label shape: {y.shape}")

Processing ECG records: 100%|██████████| 21799/21799 [4:12:48<00:00,  1.44it/s]



Completed!
X feature matrix shape: (21799, 1000, 12)
y label shape: (21799, 5)


In [12]:
save_dir = '/content/drive/MyDrive/FundamentofDS/processed/multi'

np.save(os.path.join(save_dir, 'X_multi.npy'), X)
np.save(os.path.join(save_dir, 'y_multi.npy'), y)
label_cols = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
df[['patient_id', 'age', 'sex', 'strat_fold'] + label_cols].to_csv(
    os.path.join(save_dir, 'metadata_multi.csv')
)